In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load dataset
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    'jackksoncsie/spam-email-dataset',
    'emails.csv',
)

df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [ ]:
# Check ratio
df['spam'].value_counts()

spam
0    4360
1    1368
Name: count, dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split

# Clean X
df['text'] = df['text'].str.removeprefix('Subject: ')

# Assign X, y
X = df['text']
y = df['spam']

# Split train, test
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=11, stratify=y
)

df.head()

,text,spam
0,naturally irresistible your corporate identity...,1
1,the stock trading gunslinger fanny is merrill...,1
2,unbelievable new homes made easy im wanting t...,1
3,4 color printing special request additional i...,1
4,"do not have money , get software cds from here...",1


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

# Pipeline
pipe = Pipeline([
	('preprocessor', None),
    ('model', DummyClassifier())
])

In [ ]:
# Count vectorizer params
vec_params = {
    'vec__ngram_range': [(1, 1), (1, 2), (1, 3), (2, 3)],
    'vec__min_df': [1, 2, 3, 4],
    'vec__max_df': [1, 0.95, 0.9],
    'vec__stop_words': ['english'],
    'vec__binary': [True]
}

# Additional params for TF-IDF
tfidf_params = {
    'vec__sublinear_tf': [True, False],
    'vec__norm': [None, 'l2']
}

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Combine into grid
count_grid = {
    'vec': [CountVectorizer()],
    **vec_params,
}

tfidf_grid = {
    'vec': [TfidfVectorizer()],
    **vec_params,
    **tfidf_params
}

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.neural_network import MLPClassifier

# Model grids
model_grids = [
    {
        'model': [
            LogisticRegression(
                max_iter=1000,
                class_weight='balanced',
                random_state=42
            )
        ],
        'model__C': [0.01, 0.1, 1, 10],
        'model__solver': ['lbfgs', 'saga']
    },
    {
        'model': [
            BernoulliNB()
        ],
        'model__alpha': [0.001, 0.01, 0.1]
    },
    {
        'model': [
            MLPClassifier(
                activation='relu',
                solver='adam',
                random_state=42
            )
        ],
        'model__hidden_layer_sizes': [(10,), (10, 20), (10, 20, 20)],
        'model__learning_rate_init': [0.001, 0.01, 0.1],
        'model__alpha': [0.001, 0.01, 0.1],
        'model__max_iter': [100, 200, 1000, 2000]
    }
]

In [ ]:
# Combine vec grid, model grid
count_param_grid = []
tfidf_param_grid = []

for model_grid in model_grids:
    count_param_grid.append({**count_grid, **model_grid})
    tfidf_param_grid.append({**tfidf_grid, **model_grid})

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report

for param_grid in [count_param_grid, tfidf_param_grid]:
	# Hyperparameter tuning
	search = RandomizedSearchCV(
		estimator=pipe,
		param_distributions=param_grid,
		cv=2,
		scoring='precision',
		n_jobs=-1,
		n_iter=10,
		random_state=11
	)
	search.fit(X_train, y_train)

	# Print name
	best_pipe = search.best_estimator_
	vectorizer_name = best_pipe.named_steps['vec'].__class__.__name__
	model_name = best_pipe.named_steps['model'].__class__.__name__
	print(f'----------\n{vectorizer_name} + {model_name}\n----------')

	# Evaluation
	score = search.score(X_test, y_test)

	y_pred = search.predict(X_test)
	cm = confusion_matrix(y_test, y_pred)
	cr = classification_report(y_test, y_pred)

	print(f'Score: {score}')
	print(f'\nConfusion matrix:\n {cm}')
	print(f'\nClassification report:\n {cr}')

	print(f'\nBest params: {search.best_params_}')